In [18]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader, Subset
from dataset import IMDBDataset, get_celeb_name
from model import AgeClassifier
from torch.utils.data import random_split, SubsetRandomSampler
from torch import optim, nn
from tqdm import tqdm
from unlearn import *
from utils import *
from sklearn.model_selection import train_test_split
import random

In [19]:
teacher_checkpoint_path = 'IMDB_CROP_Pretrained_50000_Samples_Teacher.pt'
forget_checkpoint_path = 'IMDB_CROP_Pretrained_50000_Samples_Forget.pt'

### Unzip Dataset

In [20]:
import tarfile
import os

if not os.path.isdir('./imdb_crop'):
    with tarfile.open('./imdb_crop.tar', 'r') as tar:
        tar.extractall('./')

In [21]:
dataset = IMDBDataset('./imdb.csv', './imdb_crop', 50000) # remove this for full dataset

train_idx, valid_idx = train_test_split(list(range(len(dataset))), test_size=0.2, random_state=42)

train_ds = Subset(dataset, train_idx)
valid_ds = Subset(dataset, valid_idx)

Balanced dataset: 49129 samples across 5 age classes
Unique celebrities: 3309
Class distribution: {0: 9129, 1: 10000, 2: 10000, 3: 10000, 4: 10000}


In [22]:
celeb_ids = torch.tensor(dataset.celeb_ids)
ages = torch.tensor(dataset.ages)

unique_celebs = torch.unique(celeb_ids)
valid_celebs = []

for celeb_id in unique_celebs:
    if torch.any(ages[celeb_ids == celeb_id] <= 13):
        valid_celebs.append(celeb_id.item())

num_celeb = 5
if len(valid_celebs) < num_celeb:
    raise ValueError(f"Only {len(valid_celebs)} celebrities have images where age <= 13, but {num_celeb} requested")

forget_celeb_ids = random.sample(valid_celebs[:10], num_celeb)

In [23]:
print('forget celeb names:')
print(*[get_celeb_name(celeb_id) for celeb_id in forget_celeb_ids], sep='\n')

forget celeb names:
['Adam Harrington']
['A.J. Trauth']
['Abigail Hargrove']
['Adal Ramones']
['Adam Richman']


In [24]:
retain_train_ds = Subset(dataset, torch.tensor([i for i in train_idx if celeb_ids[i].item() not in forget_celeb_ids]))
retain_valid_ds = Subset(dataset, torch.tensor([i for i in valid_idx if celeb_ids[i].item() not in forget_celeb_ids]))

forget_train_ds = Subset(dataset, torch.tensor([i for i in train_idx if celeb_ids[i].item() in forget_celeb_ids]))
forget_valid_ds = Subset(dataset, torch.tensor([i for i in valid_idx if celeb_ids[i].item() in forget_celeb_ids]))

In [25]:
print('dataset sizes:')
print(len(retain_train_ds), len(retain_valid_ds), len(forget_train_ds), len(forget_valid_ds), sep='\n')

dataset sizes:
39145
9786
158
40


In [26]:
device = 'cuda'

batch_size = 256
num_workers = 4

train_dl = DataLoader(train_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size, num_workers=num_workers, pin_memory=False)

retain_train_dl = DataLoader(retain_train_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle = True)
retain_valid_dl = DataLoader(retain_valid_ds, batch_size, num_workers=num_workers, pin_memory=False)

forget_train_dl = DataLoader(forget_train_ds, batch_size, num_workers=num_workers, pin_memory=False, shuffle = True)
forget_valid_dl = DataLoader(forget_valid_ds, batch_size, num_workers=num_workers, pin_memory=False)

In [27]:

full_trained_teacher = AgeClassifier(num_classes = 5, pretrained = True).to(device)

# Training 
history = fit_one_cycle(10, full_trained_teacher, train_dl, valid_dl, device = device)

# Loading
# full_trained_teacher.load_state_dict(torch.load("ResNET18_CIFAR100Super20_Pretrained_ALL_CLASSES_5_Epochs.pt", map_location = device))

# Saving
torch.save(full_trained_teacher.state_dict(), teacher_checkpoint_path)

C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torch\optim\lr_scheduler.py:62: UserWarn

Epoch [0], last_lr: 0.01000, train_loss: 1.7702, val_loss: 1.6170, val_acc: 20.3172
Epoch [1], last_lr: 0.01000, train_loss: 1.6787, val_loss: 1.6151, val_acc: 19.9961
Epoch [2], last_lr: 0.01000, train_loss: 1.6437, val_loss: 1.6554, val_acc: 20.8219
Epoch [3], last_lr: 0.01000, train_loss: 1.6208, val_loss: 1.6101, val_acc: 21.4251
Epoch [4], last_lr: 0.01000, train_loss: 1.6115, val_loss: 1.6083, val_acc: 21.3989
Epoch [5], last_lr: 0.01000, train_loss: 1.6081, val_loss: 1.6054, val_acc: 21.7718
Epoch [6], last_lr: 0.01000, train_loss: 1.5955, val_loss: 1.5864, val_acc: 23.2674
Epoch [7], last_lr: 0.01000, train_loss: 1.5805, val_loss: 1.5723, val_acc: 25.6001
Epoch [8], last_lr: 0.01000, train_loss: 1.5659, val_loss: 1.5565, val_acc: 28.4586
Epoch [9], last_lr: 0.01000, train_loss: 1.5534, val_loss: 1.5557, val_acc: 28.2521


In [28]:
evaluate(full_trained_teacher, retain_valid_dl, device)

{'Loss': 1.555640459060669,
 'Acc': 28.17888069152832,
 'Distribution': {'predictions': [9.687308311462402,
   31.759654998779297,
   2.5240137577056885,
   11.955855369567871,
   44.07316589355469],
  'labels': [18.781932830810547,
   20.03883171081543,
   20.171672821044922,
   20.897201538085938,
   20.110361099243164],
  'pred_counts': [948, 3108, 247, 1170, 4313],
  'label_counts': [1838, 1961, 1974, 2045, 1968]}}

In [29]:
evaluate(full_trained_teacher, forget_valid_dl, device)

{'Loss': 1.5814099311828613,
 'Acc': 32.5,
 'Distribution': {'predictions': [5.0, 65.0, 2.5, 2.5, 25.0],
  'labels': [57.5, 42.5, 0.0, 0.0, 0.0],
  'pred_counts': [2, 26, 1, 1, 10],
  'label_counts': [23, 17, 0, 0, 0]}}

### Forget

In [ ]:
model = AgeClassifier(num_classes = 5, pretrained = False).to(device)
unlearning_teacher = AgeClassifier(num_classes = 5, pretrained = False).to(device)

# Training
model.load_state_dict(torch.load(teacher_checkpoint_path, map_location = device))
blindspot_unlearner(model = model, unlearning_teacher = unlearning_teacher, full_trained_teacher = full_trained_teacher, 
                    retain_data = retain_train_ds, forget_data = forget_train_ds, epochs = 1, lr = 0.0001, 
                    batch_size = batch_size, num_workers = num_workers, device = device)

# Loading
# model.load_state_dict(torch.load(forget_checkpoint_path, map_location = device))

# Saving
torch.save(model.state_dict(), forget_checkpoint_path)

C:\Users\Foopy\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Epoch 1 Unlearning Loss 0.004543218296021223
Epoch 2 Unlearning Loss 0.003644597949460149


In [31]:
evaluate(model, retain_valid_dl, device)

{'Loss': 1.5559821128845215,
 'Acc': 27.8984317779541,
 'Distribution': {'predictions': [8.828939437866211,
   33.27202224731445,
   2.3400776386260986,
   12.18066692352295,
   43.3782958984375],
  'labels': [18.781932830810547,
   20.03883171081543,
   20.171672821044922,
   20.897201538085938,
   20.110361099243164],
  'pred_counts': [864, 3256, 229, 1192, 4245],
  'label_counts': [1838, 1961, 1974, 2045, 1968]}}

In [32]:
evaluate(model, forget_valid_dl, device)

{'Loss': 1.5831215381622314,
 'Acc': 32.5,
 'Distribution': {'predictions': [5.0, 65.0, 2.5, 2.5, 25.0],
  'labels': [57.5, 42.5, 0.0, 0.0, 0.0],
  'pred_counts': [2, 26, 1, 1, 10],
  'label_counts': [23, 17, 0, 0, 0]}}